# Red Neuronal de Grafos + LSTM -- Sistema Financiero Ecuador (v2)
Predecir la morosidad total de cada entidad financiera (bancos y cooperativas) usando una red heterogenea de grafos combinada con LSTM.

**Cambios v2:**
- Integracion de indicadores macroeconomicos del Ecuador
- Filtro desde 2016-01 (bancos + cooperativas)
- Fix: bug de aristas vacias en k-NN
- Fix: Dataset usa variable de instancia en vez de global (se aumenta self.)
- LSTM per-nodo con pesos compartidos (rastrea trayectoria individual)
- LSTM de contexto global (pooling + macro)
- Perdida BCE (log-loss) en vez de MSE — morosidad es proporcion [0,1]
- Salida con sigmoid, target sin escalar

**Arquitectura:**
- Nodos tipo `banco` y tipo `cooperativa` con features propias
- Aristas construidas dinamicamente por periodo sobre similitud coseno
- Capas HGTConv (Heterogeneous Graph Transformer)
- LSTM per-nodo (pesos compartidos): rastrea la evolucion individual de cada entidad
- LSTM global: captura dinamica del sistema + entorno macro
- Concatenacion [h_individual, h_global] -> FC -> sigmoid -> prediccion de morosidad


## Instalación

In [1]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

!pip install -q torch-geometric
!pip install -q scikit-learn pandas openpyxl matplotlib seaborn networkx tqdm

print('-- Instalación completa')

PyTorch version : 2.10.0+cpu
CUDA disponible : False
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.3 MB/s eta 0:00:00
-- Instalación completa


## Montar Google Drive y cargar datos

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np

DRIVE_PATH = '/content/drive/MyDrive/GNN_SF/'

SB   = pd.read_excel(DRIVE_PATH + 'data/balance_indicadores_SB_limpio.xlsx', index_col=[0])
SEPS = pd.read_excel(DRIVE_PATH + 'data/balance_indicadores_SEPS_limpio.xlsx', index_col=[0])

# ─── Cargar indicadores macroeconómicos ────────────────────────────────────
MACRO = pd.read_excel(DRIVE_PATH + 'data/indicadoresMacro.xlsx')

print(f'SB    shape : {SB.shape}')
print(f'SEPS  shape : {SEPS.shape}')
print(f'MACRO shape : {MACRO.shape}')
print()
print('─── Columnas MACRO ───')
print(MACRO.columns.tolist())
print()
print(MACRO.head())

SB    shape : (5399, 137)
SEPS  shape : (4178, 127)
MACRO shape : (129, 5)

─── Columnas MACRO ───
['fechaCorte', 'tasaPasiva', 'riesgoPaisSobreMil', 'tasaPIB', 'petroleoSobreCien']

  fechaCorte  tasaPasiva  riesgoPaisSobreMil   tasaPIB  petroleoSobreCien
0    2015-01      0.0522            0.933903 -0.032898             0.4153
1    2015-02      0.0532            0.778036 -0.032898             0.4157
2    2015-03      0.0531            0.799516 -0.032898             0.4291
3    2015-04      0.0539            0.782733 -0.044700             0.5531
4    2015-05      0.0551            0.670548 -0.044700             0.5667


## Exploración, limpieza y filtro temporal (>= 2016-01)

In [4]:
# ─── Asegurar tipo datetime ────────────────────────────────────────────────
SB['fechaCorte']    = pd.to_datetime(SB['fechaCorte'])
SEPS['fechaCorte']  = pd.to_datetime(SEPS['fechaCorte'])
MACRO['fechaCorte'] = pd.to_datetime(MACRO['fechaCorte'])

# ─── FILTRO: solo desde 2016-01 (ahí ya hay bancos Y cooperativas) ─────────
FECHA_INICIO = pd.Timestamp('2016-01-01')

SB   = SB[SB['fechaCorte'] >= FECHA_INICIO].copy()
SEPS = SEPS[SEPS['fechaCorte'] >= FECHA_INICIO].copy()
MACRO = MACRO[MACRO['fechaCorte'] >= FECHA_INICIO].copy()

# Etiquetar tipo de entidad
SB['tipo']   = 'banco'
SEPS['tipo'] = 'cooperativa'

# Verificar columnas clave
for nombre, df in [('SB', SB), ('SEPS', SEPS)]:
    print(f'=== {nombre} ===')
    print(f"  Columna 'razonSocial' presente : {'razonSocial' in df.columns}")
    print(f"  Columna 'fechaCorte'  presente : {'fechaCorte'  in df.columns}")
    print(f"  Columna 'morosidad_total' presente : {'morosidad_total' in df.columns}")
    print(f"  Períodos únicos       : {df['fechaCorte'].nunique()}")
    print(f"  Entidades únicas      : {df['razonSocial'].nunique()}")
    print(f"  Rango fechas          : {df['fechaCorte'].min()} → {df['fechaCorte'].max()}")
    print()

print(f'=== MACRO ===')
print(f'  Períodos : {MACRO["fechaCorte"].nunique()}')
print(f'  Rango    : {MACRO["fechaCorte"].min()} → {MACRO["fechaCorte"].max()}')
print(f'  Nulos    :')
print(MACRO.isnull().sum())

# Revisar valores nulos en morosidad_total
print(f'\nNulos en morosidad_total:')
print(f'  SB   : {SB["morosidad_total"].isna().sum()}')
print(f'  SEPS : {SEPS["morosidad_total"].isna().sum()}')

=== SB ===
  Columna 'razonSocial' presente : True
  Columna 'fechaCorte'  presente : True
  Columna 'morosidad_total' presente : True
  Períodos únicos       : 117
  Entidades únicas      : 22
  Rango fechas          : 2016-01-31 00:00:00 → 2025-09-30 00:00:00

=== SEPS ===
  Columna 'razonSocial' presente : True
  Columna 'fechaCorte'  presente : True
  Columna 'morosidad_total' presente : True
  Períodos únicos       : 117
  Entidades únicas      : 71
  Rango fechas          : 2016-01-31 00:00:00 → 2025-09-30 00:00:00

=== MACRO ===
  Períodos : 117
  Rango    : 2016-01-01 00:00:00 → 2025-09-01 00:00:00
  Nulos    :
fechaCorte            0
tasaPasiva            0
riesgoPaisSobreMil    0
tasaPIB               0
petroleoSobreCien     0
dtype: int64

Nulos en morosidad_total:
  SB   : 0
  SEPS : 0


## Preparar indicadores macro

In [5]:
# --- Definir features macro -----------------------------------------------
FEATURES_MACRO = ['tasaPasiva', 'riesgoPaisSobreMil', 'tasaPIB', 'petroleoSobreCien']

DIM_MACRO = len(FEATURES_MACRO)
print(f'Dimension macro: {DIM_MACRO}')

# Crear diccionario fecha -> vector macro
MACRO_indexed = MACRO.set_index('fechaCorte')[FEATURES_MACRO].copy()
MACRO_indexed = MACRO_indexed.replace([np.inf, -np.inf], np.nan).fillna(method='ffill').fillna(0)

print(f'\nEstadisticas macro (post-filtro):')
print(MACRO_indexed.describe())


Dimension macro: 4

Estadisticas macro (post-filtro):
       tasaPasiva  riesgoPaisSobreMil     tasaPIB  petroleoSobreCien
count  117.000000          117.000000  117.000000         117.000000
mean     0.060997            1.156093    0.032899           0.576857
std      0.009409            0.728921    0.074115           0.167700
min      0.048000            0.452065   -0.214257           0.140300
25%      0.055100            0.711300   -0.003759           0.447900
50%      0.059100            0.896100    0.034975           0.585300
75%      0.065800            1.362600    0.065767           0.672900
max      0.084500            5.078333    0.244928           1.044000


/tmp/ipykernel_996/1526120288.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  MACRO_indexed = MACRO_indexed.replace([np.inf, -np.inf], np.nan).fillna(method='ffill').fillna(0)


## Definición de features por tipo de nodo

In [6]:
# ─── Features COMUNES (disponibles en SB y SEPS) ──────────────────────────
FEATURES_COMUNES = [
    'comercial_priorit_ndi_vs_cartera',
    'consumo_priorit_ndi_vs_cartera',
    'inmobiliario_ndi_vs_cartera',
    'micro_ndi_vs_cartera',
    'comercial_priorit_vencida_vs_cartera',
    'consumo_priorit_vencida_vs_cartera',
    'inmobiliario_vencido_vs_cartera',
    'micro_vencido_vs_cartera',
    'provisiones_cred_incobr_vs_cartera',
    'refinanciada_vs_cartera',
    'reestructurada_vs_cartera',
    'cxc_inversion_vs_cxc',
    'cxc_cartera_creditos_vs_cxc',
    'depreciacion_acum_vs_propiedad',
    'otros_activos_vs_activo',
    'inv_acciones_y_particip_vs_activo',
    'en_otras_inst_finan_vs_activo',
    'depositos_a_la_vista_vs_pasivo',
    'depositos_a_plazo_vs_pasivo',
    'oblig_con_ifis_y_sector_f_vs_pasivo',
    'oblig_ifis_sector_publico_vs_pasivo',
    'otros_pasivos_vs_pasivos',
    'capital_social_vs_patrimonio',
    'reservas_vs_patrimonio',
    'resultados_vs_patrimonio',
    'gastos_vs_patrimonio',
    'provisiones_vs_patrimonio',
    'gastos_operacion_vs_patrimonio',
    'fondos_disponibles_vs_activos',
    'caja_vs_fondos_disp',
    'bancos_y_otras_inst_finan_vs_fondos_disp',
    'banco_central_vs_bancos_y_otr',
    'bancos_locales_vs_bancos_y_otr',
    'comercial_prioritario_xvencer_vs_cartera',
    'consumo_prioritario_xvencer_vs_cartera',
    'inmobiliario_xvencer_vs_cartera',
    'micro_xvencer_vs_cartera',
    'depositos_vs_ingresos',
    'intereses_descuentos_inv_vs_ingresos',
    'intereses_cartera_creditos_vs_ingresos',
    'comisiones_vs_ingresos',
    'ingresos_servicios_vs_ingresos',
    'otros_ingresos_vs_ingresos',
    'cuentas_contingentes_vs_activo',
]

# ─── Features EXCLUSIVAS ──────────────────────────────────────────────────
import re

def es_columna_numerica_original(col):
    return bool(re.fullmatch(r'\d+', col))

cols_excluir = set(['razonSocial', 'fechaCorte', 'tipo', 'morosidad_total'] + FEATURES_COMUNES)

FEATURES_SB = [
    c for c in SB.columns
    if c not in cols_excluir
    and not es_columna_numerica_original(c)
]

FEATURES_SEPS = [
    c for c in SEPS.columns
    if c not in cols_excluir
    and not es_columna_numerica_original(c)
]

FEATURES_NODO_BANCO = FEATURES_COMUNES + FEATURES_SB
FEATURES_NODO_COOP  = FEATURES_COMUNES + FEATURES_SEPS

TARGET = 'morosidad_total'

print(f'Features comunes        : {len(FEATURES_COMUNES)}')
print(f'Features exclusivas SB  : {len(FEATURES_SB)}')
print(f'Features exclusivas SEPS: {len(FEATURES_SEPS)}')
print(f'Dim nodo banco          : {len(FEATURES_NODO_BANCO)}')
print(f'Dim nodo cooperativa    : {len(FEATURES_NODO_COOP)}')
print(f'Dim macro               : {DIM_MACRO}')

Features comunes        : 44
Features exclusivas SB  : 38
Features exclusivas SEPS: 28
Dim nodo banco          : 82
Dim nodo cooperativa    : 72
Dim macro               : 4


In [7]:
# Verificar que todos los features existen en los dataframes
faltantes_SB   = [f for f in FEATURES_NODO_BANCO if f not in SB.columns]
faltantes_SEPS = [f for f in FEATURES_NODO_COOP  if f not in SEPS.columns]

if faltantes_SB:
    print(f'ADVERTENCIA - Features faltantes en SB   : {faltantes_SB}')
else:
    print('OK - Todos los features de banco estan en SB')

if faltantes_SEPS:
    print(f'ADVERTENCIA - Features faltantes en SEPS : {faltantes_SEPS}')
else:
    print('OK - Todos los features de cooperativa estan en SEPS')


OK - Todos los features de banco estan en SB
OK - Todos los features de cooperativa estan en SEPS


## Universo global de entidades y normalización

In [8]:
from sklearn.preprocessing import RobustScaler

# ─── Universo global (solo entidades que existen desde 2016-01) ────────────
bancos_universo = sorted(SB['razonSocial'].unique())
coops_universo  = sorted(SEPS['razonSocial'].unique())

banco2idx = {b: i for i, b in enumerate(bancos_universo)}
coop2idx  = {c: i for i, c in enumerate(coops_universo)}

N_BANCOS = len(bancos_universo)
N_COOPS  = len(coops_universo)

print(f'Bancos en el universo      : {N_BANCOS}')
print(f'Cooperativas en el universo: {N_COOPS}')
print(f'Total nodos                : {N_BANCOS + N_COOPS}')

Bancos en el universo      : 22
Cooperativas en el universo: 71
Total nodos                : 93


In [9]:
# ─── Split temporal ───────────────────────────────────────────────────────
periodos = sorted(set(SB['fechaCorte'].unique()) | set(SEPS['fechaCorte'].unique()))
print(f'Períodos totales: {len(periodos)}')
print(f'Primer período  : {periodos[0]}')
print(f'Último período  : {periodos[-1]}')

n = len(periodos)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)

periodos_train = periodos[:n_train]
periodos_val   = periodos[n_train:n_train + n_val]
periodos_test  = periodos[n_train + n_val:]

print(f'\nSplit temporal:')
print(f'  Train : {len(periodos_train)} períodos ({periodos_train[0]} → {periodos_train[-1]})')
print(f'  Val   : {len(periodos_val)}  períodos ({periodos_val[0]}  → {periodos_val[-1]})')
print(f'  Test  : {len(periodos_test)}  períodos ({periodos_test[0]}  → {periodos_test[-1]})')

Períodos totales: 117
Primer período  : 2016-01-31 00:00:00
Último período  : 2025-09-30 00:00:00

Split temporal:
  Train : 93 períodos (2016-01-31 00:00:00 → 2023-09-30 00:00:00)
  Val   : 11  períodos (2023-10-31 00:00:00  → 2024-08-31 00:00:00)
  Test  : 13  períodos (2024-09-30 00:00:00  → 2025-09-30 00:00:00)


In [ ]:
# ─── Normalización ────────────────────────────────────────────────────────
SB_train   = SB[SB['fechaCorte'].isin(periodos_train)]
SEPS_train = SEPS[SEPS['fechaCorte'].isin(periodos_train)]

scaler_banco = RobustScaler()
scaler_coop  = RobustScaler()
scaler_macro = RobustScaler()  # scaler para indicadores macro
# NOTA: NO se escala el target (morosidad_total) porque ya está en [0,1]
#       y usaremos BCE como función de pérdida

def preparar_features(df, features):
    X = df[features].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    col_medians = X.median()
    col_medians = col_medians.fillna(0)
    X = X.fillna(col_medians)
    X = X.fillna(0)
    return X

scaler_banco.fit(preparar_features(SB_train, FEATURES_NODO_BANCO))
scaler_coop.fit(preparar_features(SEPS_train, FEATURES_NODO_COOP))

# Escalar indicadores macro (solo con datos de train)
macro_train = MACRO_indexed.loc[MACRO_indexed.index.isin(periodos_train)]
scaler_macro.fit(macro_train)

# Crear diccionario de vectores macro escalados por fecha
macro_scaled_dict = {}
for fecha in periodos:
    if fecha in MACRO_indexed.index:
        vec = scaler_macro.transform(MACRO_indexed.loc[[fecha]]).flatten()
    else:
        fechas_disponibles = MACRO_indexed.index[MACRO_indexed.index <= fecha]
        if len(fechas_disponibles) > 0:
            vec = scaler_macro.transform(MACRO_indexed.loc[[fechas_disponibles[-1]]]).flatten()
        else:
            vec = np.zeros(DIM_MACRO)
    macro_scaled_dict[fecha] = vec.astype(np.float32)

print('Escaladores ajustados sobre datos de entrenamiento')
print(f'  scaler_banco : {scaler_banco.n_features_in_} features')
print(f'  scaler_coop  : {scaler_coop.n_features_in_} features')
print(f'  scaler_macro : {scaler_macro.n_features_in_} features')
print(f'  target (morosidad): SIN escalar — ya está en [0,1], se usa BCE loss')


## Construcción de snapshots y aristas dinámicas

In [ ]:
import torch
from torch_geometric.data import HeteroData
from sklearn.metrics.pairwise import cosine_similarity

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


def construir_aristas_knn(X: np.ndarray, k: int = 5):
    """
    Construye aristas k-NN basadas en similitud coseno.
    FIX: manejo correcto de arrays vacíos tras deduplicación.
    """
    n = len(X)
    if n <= 1:
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0, 1), dtype=np.float32)

    k = min(k, n - 1)
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, -2)

    top_k_idx = np.argsort(sim, axis=1)[:, -k:]
    filas = np.repeat(np.arange(n), k)
    cols  = top_k_idx.flatten()

    similitud_vals = sim[filas, cols].reshape(-1, 1)

    # Aristas bidireccionales
    edge_index = np.stack([np.concatenate([filas, cols]),
                           np.concatenate([cols, filas])], axis=0)
    edge_attr  = np.vstack([similitud_vals, similitud_vals])

    # Deduplicar
    edges_set = set(map(tuple, edge_index.T.tolist()))

    if len(edges_set) == 0:
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0, 1), dtype=np.float32)

    edge_index = np.array(list(edges_set), dtype=np.int64).T
    edge_attr = sim[edge_index[0], edge_index[1]].reshape(-1, 1)

    return edge_index.astype(np.int64), edge_attr.astype(np.float32)


def build_snapshot(fecha, SB_df, SEPS_df, K_INTRA=5, K_INTER=3):
    """
    Construye un HeteroData para un período dado.
    Target (morosidad) se almacena RAW en [0,1] — sin escalar.
    """
    snapshot = HeteroData()

    sb_t   = SB_df[SB_df['fechaCorte'] == fecha].set_index('razonSocial')
    seps_t = SEPS_df[SEPS_df['fechaCorte'] == fecha].set_index('razonSocial')

    bancos_activos = [b for b in bancos_universo if b in sb_t.index]
    coops_activas  = [c for c in coops_universo  if c in seps_t.index]

    # ── Features de nodo bancos ────────────────────────────────────────────
    X_banco_full = np.zeros((N_BANCOS, len(FEATURES_NODO_BANCO)), dtype=np.float32)
    mask_banco   = np.zeros(N_BANCOS, dtype=bool)
    y_banco      = np.zeros(N_BANCOS, dtype=np.float32)

    if len(bancos_activos) > 0:
        X_raw = preparar_features(sb_t.loc[bancos_activos], FEATURES_NODO_BANCO).values
        X_scaled = scaler_banco.transform(X_raw)
        for i, b in enumerate(bancos_activos):
            idx = banco2idx[b]
            X_banco_full[idx] = X_scaled[i]
            mask_banco[idx]   = True
            val = sb_t.loc[b, 'morosidad_total'] if pd.notna(sb_t.loc[b, 'morosidad_total']) else 0.0
            y_banco[idx] = np.clip(val, 0.0, 1.0)  # asegurar [0,1] para BCE

    # ── Features de nodo cooperativas ─────────────────────────────────────
    X_coop_full = np.zeros((N_COOPS, len(FEATURES_NODO_COOP)), dtype=np.float32)
    mask_coop   = np.zeros(N_COOPS, dtype=bool)
    y_coop      = np.zeros(N_COOPS, dtype=np.float32)

    if len(coops_activas) > 0:
        X_raw = preparar_features(seps_t.loc[coops_activas], FEATURES_NODO_COOP).values
        X_scaled = scaler_coop.transform(X_raw)
        for i, c in enumerate(coops_activas):
            idx = coop2idx[c]
            X_coop_full[idx] = X_scaled[i]
            mask_coop[idx]   = True
            val = seps_t.loc[c, 'morosidad_total'] if pd.notna(seps_t.loc[c, 'morosidad_total']) else 0.0
            y_coop[idx] = np.clip(val, 0.0, 1.0)

    snapshot['banco'].x          = torch.tensor(X_banco_full, dtype=torch.float)
    snapshot['banco'].mask       = torch.tensor(mask_banco,   dtype=torch.bool)
    snapshot['banco'].y          = torch.tensor(y_banco,      dtype=torch.float)
    snapshot['cooperativa'].x    = torch.tensor(X_coop_full,  dtype=torch.float)
    snapshot['cooperativa'].mask = torch.tensor(mask_coop,    dtype=torch.bool)
    snapshot['cooperativa'].y    = torch.tensor(y_coop,       dtype=torch.float)

    # ── Vector macro del período ──────────────────────────────────────────
    snapshot['macro'] = torch.tensor(macro_scaled_dict[fecha], dtype=torch.float)

    # ── Aristas intra-banco ────────────────────────────────────────────────
    if len(bancos_activos) >= 2:
        idx_activos_b = [banco2idx[b] for b in bancos_activos]
        X_sub = X_banco_full[idx_activos_b]
        ei_sub, ea = construir_aristas_knn(X_sub, k=K_INTRA)
        idx_activos_b_arr = np.array(idx_activos_b)
        ei_global = idx_activos_b_arr[ei_sub]
        snapshot['banco', 'similar_a', 'banco'].edge_index = torch.tensor(ei_global, dtype=torch.long)
        snapshot['banco', 'similar_a', 'banco'].edge_attr  = torch.tensor(ea, dtype=torch.float)
    else:
        snapshot['banco', 'similar_a', 'banco'].edge_index = torch.zeros((2, 0), dtype=torch.long)
        snapshot['banco', 'similar_a', 'banco'].edge_attr  = torch.zeros((0, 1), dtype=torch.float)

    # ── Aristas intra-cooperativa ──────────────────────────────────────────
    if len(coops_activas) >= 2:
        idx_activos_c = [coop2idx[c] for c in coops_activas]
        X_sub = X_coop_full[idx_activos_c]
        ei_sub, ea = construir_aristas_knn(X_sub, k=K_INTRA)
        idx_activos_c_arr = np.array(idx_activos_c)
        ei_global = idx_activos_c_arr[ei_sub]
        snapshot['cooperativa', 'similar_a', 'cooperativa'].edge_index = torch.tensor(ei_global, dtype=torch.long)
        snapshot['cooperativa', 'similar_a', 'cooperativa'].edge_attr  = torch.tensor(ea, dtype=torch.float)
    else:
        snapshot['cooperativa', 'similar_a', 'cooperativa'].edge_index = torch.zeros((2, 0), dtype=torch.long)
        snapshot['cooperativa', 'similar_a', 'cooperativa'].edge_attr  = torch.zeros((0, 1), dtype=torch.float)

    # ── Aristas inter (banco <-> cooperativa) ─────────────────────────────
    if len(bancos_activos) >= 1 and len(coops_activas) >= 1:
        idx_b = [banco2idx[b] for b in bancos_activos]
        idx_c = [coop2idx[c]  for c in coops_activas]

        X_b_common = X_banco_full[idx_b][:, :len(FEATURES_COMUNES)]
        X_c_common = X_coop_full[idx_c][:, :len(FEATURES_COMUNES)]

        sim_cross = cosine_similarity(X_b_common, X_c_common)

        k_inter_b = min(K_INTER, len(coops_activas))
        top_k_b   = np.argsort(sim_cross, axis=1)[:, -k_inter_b:]
        b_rows    = np.repeat(np.arange(len(bancos_activos)), k_inter_b)
        c_cols    = top_k_b.flatten()

        ei_bc = np.stack([
            np.array(idx_b)[b_rows],
            np.array(idx_c)[c_cols]
        ], axis=0)

        k_inter_c  = min(K_INTER, len(bancos_activos))
        top_k_c    = np.argsort(sim_cross.T, axis=1)[:, -k_inter_c:]
        c_rows     = np.repeat(np.arange(len(coops_activas)), k_inter_c)
        b_cols     = top_k_c.flatten()

        ei_cb = np.stack([
            np.array(idx_b)[b_cols],
            np.array(idx_c)[c_rows]
        ], axis=0)

        ei_bc_combined = np.concatenate([ei_bc, ei_cb], axis=1)
        ei_bc_combined = np.unique(ei_bc_combined, axis=1)

        idx_b_arr = np.array(idx_b)
        idx_c_arr = np.array(idx_c)
        b_local   = np.searchsorted(idx_b_arr, ei_bc_combined[0])
        c_local   = np.searchsorted(idx_c_arr, ei_bc_combined[1])
        ea_bc     = sim_cross[b_local, c_local].reshape(-1, 1).astype(np.float32)

        snapshot['banco', 'relacionado_con', 'cooperativa'].edge_index = torch.tensor(ei_bc_combined,       dtype=torch.long)
        snapshot['banco', 'relacionado_con', 'cooperativa'].edge_attr  = torch.tensor(ea_bc,                dtype=torch.float)
        snapshot['cooperativa', 'relacionado_con', 'banco'].edge_index = torch.tensor(ei_bc_combined[[1,0]], dtype=torch.long)
        snapshot['cooperativa', 'relacionado_con', 'banco'].edge_attr  = torch.tensor(ea_bc,                dtype=torch.float)
    else:
        for rel in [('banco','relacionado_con','cooperativa'), ('cooperativa','relacionado_con','banco')]:
            snapshot[rel].edge_index = torch.zeros((2,0), dtype=torch.long)
            snapshot[rel].edge_attr  = torch.zeros((0,1), dtype=torch.float)

    snapshot['fecha'] = fecha
    return snapshot

print('Funciones de construcción de snapshots listas')


In [ ]:
# ─── Construir todos los snapshots ────────────────────────────────────────
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

K_INTRA = 5
K_INTER = 3

snapshots = {}
for fecha in tqdm(periodos, desc='Construyendo snapshots'):
    snapshots[fecha] = build_snapshot(fecha, SB, SEPS, K_INTRA, K_INTER)

print(f'\n{len(snapshots)} snapshots construidos')

ejemplo = snapshots[periodos[-1]]
print(f'\nEjemplo snapshot ({periodos[-1]}):')
print(f'  banco.x shape      : {ejemplo["banco"].x.shape}')
print(f'  cooperativa.x shape: {ejemplo["cooperativa"].x.shape}')
print(f'  bancos activos     : {ejemplo["banco"].mask.sum().item()}')
print(f'  coops activas      : {ejemplo["cooperativa"].mask.sum().item()}')
print(f'  macro vector       : {ejemplo["macro"].shape}')
print(f'  aristas b-b        : {ejemplo["banco","similar_a","banco"].edge_index.shape[1]}')
print(f'  aristas c-c        : {ejemplo["cooperativa","similar_a","cooperativa"].edge_index.shape[1]}')
print(f'  aristas b-c        : {ejemplo["banco","relacionado_con","cooperativa"].edge_index.shape[1]}')

## Dataset con ventanas deslizantes

In [ ]:
from torch.utils.data import Dataset, DataLoader

SEQ_LEN = 6

class FinancialGraphDataset(Dataset):
    """
    Cada muestra es una secuencia de SEQ_LEN snapshots (X)
    y el snapshot siguiente como target (y).
    FIX: usa snapshots_dict de instancia, no variable global.
    """
    def __init__(self, lista_periodos, snapshots_dict, seq_len):
        self.seq_len = seq_len
        self.snapshots_dict = snapshots_dict  # FIX: guardar referencia
        self.samples = []
        for i in range(len(lista_periodos) - seq_len):
            seq_fechas   = lista_periodos[i : i + seq_len]
            target_fecha = lista_periodos[i + seq_len]
            self.samples.append((seq_fechas, target_fecha))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq_fechas, target_fecha = self.samples[idx]
        seq_snapshots = [self.snapshots_dict[f] for f in seq_fechas]  # FIX
        target_snap   = self.snapshots_dict[target_fecha]              # FIX
        return seq_snapshots, target_snap


periodos_val_ext  = periodos[n_train - SEQ_LEN : n_train + n_val]
periodos_test_ext = periodos[n_train + n_val - SEQ_LEN :]

train_dataset = FinancialGraphDataset(periodos_train, snapshots, SEQ_LEN)
val_dataset   = FinancialGraphDataset(periodos_val_ext,  snapshots, SEQ_LEN)
test_dataset  = FinancialGraphDataset(periodos_test_ext, snapshots, SEQ_LEN)

print(f'Train samples : {len(train_dataset)}')
print(f'Val   samples : {len(val_dataset)}')
print(f'Test  samples : {len(test_dataset)}')

def collate_fn(batch):
    return batch

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False, collate_fn=collate_fn)

print('DataLoaders listos')

## Modelo HeteroGNN + LSTM per-nodo con indicadores macro

**Arquitectura de dos LSTMs:**

1. **LSTM per-nodo (pesos compartidos)**: Despues del GNN, cada nodo tiene un embedding de `hidden_gnn` dimensiones en cada timestep. A lo largo de SEQ_LEN timesteps se acumula una secuencia por nodo. Un unico LSTM (compartido entre todos los nodos) procesa cada secuencia individualmente. Asi se rastrea la trayectoria temporal de cada entidad sin explotar los parametros — los pesos son los mismos para todos, el LSTM aprende "como evoluciona una entidad financiera en general".

2. **LSTM de contexto global**: En paralelo, un segundo LSTM procesa la representacion agregada del grafo (pooling mean+max por tipo) concatenada con el embedding macro. Esto captura la dinamica del sistema financiero como un todo y el entorno macroeconomico.

3. **Prediccion per-nodo**: Para cada nodo se concatena su hidden state individual (del LSTM per-nodo) con el contexto global (del LSTM de contexto), y eso pasa por una cabeza FC con sigmoid para predecir morosidad en [0,1].

4. **Perdida BCE**: Como la morosidad es una proporcion en [0,1], se usa binary cross-entropy con logits como funcion de perdida.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HGTConv, Linear

NODE_TYPES = ['banco', 'cooperativa']
EDGE_TYPES = [
    ('banco',        'similar_a',       'banco'),
    ('cooperativa',  'similar_a',       'cooperativa'),
    ('banco',        'relacionado_con', 'cooperativa'),
    ('cooperativa',  'relacionado_con', 'banco'),
]
METADATA = (NODE_TYPES, EDGE_TYPES)


class HeteroGraphLSTM_v2(nn.Module):
    """
    Arquitectura con dos LSTMs:
      - LSTM per-nodo (compartido): rastrea trayectoria individual de cada entidad
      - LSTM global: captura dinamica del sistema + macro
    Salida: logits (sin sigmoid). La perdida aplica sigmoid internamente.
    """
    def __init__(
        self,
        dim_banco:     int,
        dim_coop:      int,
        dim_macro:     int,
        hidden_gnn:    int = 32,
        n_gnn_layers:  int = 2,
        n_heads:       int = 4,
        hidden_lstm_node:   int = 64,   # LSTM per-nodo
        hidden_lstm_global: int = 64,   # LSTM contexto global
        n_lstm_layers: int = 2,
        hidden_macro:  int = 16,
        dropout:       float = 0.2,
        n_bancos:      int = N_BANCOS,
        n_coops:       int = N_COOPS,
    ):
        super().__init__()
        self.hidden_gnn         = hidden_gnn
        self.n_bancos           = n_bancos
        self.n_coops            = n_coops
        self.n_gnn_layers       = n_gnn_layers
        self.hidden_lstm_node   = hidden_lstm_node
        self.hidden_lstm_global = hidden_lstm_global

        # -- Proyecciones de entrada ----------------------------------------
        self.proj_banco = nn.Sequential(
            Linear(dim_banco, hidden_gnn),
            nn.LayerNorm(hidden_gnn),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.proj_coop = nn.Sequential(
            Linear(dim_coop, hidden_gnn),
            nn.LayerNorm(hidden_gnn),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # -- Capas HGTConv --------------------------------------------------
        self.hgt_layers = nn.ModuleList([
            HGTConv(
                in_channels=hidden_gnn,
                out_channels=hidden_gnn,
                metadata=METADATA,
                heads=n_heads,
            )
            for _ in range(n_gnn_layers)
        ])
        self.layer_norms = nn.ModuleList([
            nn.ModuleDict({
                'banco': nn.LayerNorm(hidden_gnn),
                'cooperativa': nn.LayerNorm(hidden_gnn)
            })
            for _ in range(n_gnn_layers)
        ])
        self.dropout = nn.Dropout(dropout)

        # -- MLP para indicadores macro -------------------------------------
        self.macro_mlp = nn.Sequential(
            nn.Linear(dim_macro, hidden_macro),
            nn.LayerNorm(hidden_macro),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # -- LSTM per-nodo (pesos compartidos) ------------------------------
        # Input: embedding del nodo despues del GNN (hidden_gnn)
        # Procesa la secuencia temporal de CADA nodo individualmente
        # Un unico LSTM aplicado a todos los nodos en paralelo via batch
        self.lstm_node = nn.LSTM(
            input_size=hidden_gnn,
            hidden_size=hidden_lstm_node,
            num_layers=n_lstm_layers,
            batch_first=True,
            dropout=dropout if n_lstm_layers > 1 else 0.0
        )

        # -- LSTM de contexto global ----------------------------------------
        # Input: pooling del grafo + embedding macro
        # Captura la dinamica del sistema financiero + entorno macro
        global_input_dim = 4 * hidden_gnn + hidden_macro
        self.lstm_global = nn.LSTM(
            input_size=global_input_dim,
            hidden_size=hidden_lstm_global,
            num_layers=n_lstm_layers,
            batch_first=True,
            dropout=dropout if n_lstm_layers > 1 else 0.0
        )

        # -- Cabezas de prediccion per-nodo ---------------------------------
        # Input: h_individual (hidden_lstm_node) + h_global (hidden_lstm_global)
        pred_input_dim = hidden_lstm_node + hidden_lstm_global

        self.fc_banco = nn.Sequential(
            nn.Linear(pred_input_dim, pred_input_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(pred_input_dim // 2, 1)
        )
        self.fc_coop = nn.Sequential(
            nn.Linear(pred_input_dim, pred_input_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(pred_input_dim // 2, 1)
        )

    def _pool_masked(self, x, mask):
        """Pooling mean+max solo sobre nodos activos. Para el LSTM global."""
        if mask.sum() == 0:
            return torch.zeros(2 * self.hidden_gnn, device=x.device)
        x_active = x[mask]
        pool_mean = x_active.mean(dim=0)
        pool_max  = x_active.max(dim=0).values
        return torch.cat([pool_mean, pool_max])

    def forward(self, seq_snapshots: list):
        """
        seq_snapshots: lista de HeteroData de longitud SEQ_LEN
        Retorna logits (sin sigmoid):
          logits_banco : (N_BANCOS,)
          logits_coop  : (N_COOPS,)
        """
        T = len(seq_snapshots)

        # Acumuladores de embeddings per-nodo a lo largo del tiempo
        banco_embs = []   # lista de T tensores de shape (N_BANCOS, hidden_gnn)
        coop_embs  = []   # lista de T tensores de shape (N_COOPS, hidden_gnn)
        global_repr_seq = []  # representacion global por timestep

        for snap in seq_snapshots:
            # -- Proyeccion de entrada --------------------------------------
            x_dict = {
                'banco':       self.proj_banco(snap['banco'].x.to(device)),
                'cooperativa': self.proj_coop(snap['cooperativa'].x.to(device)),
            }
            edge_index_dict = {
                etype: snap[etype].edge_index.to(device)
                for etype in EDGE_TYPES
                if snap[etype].edge_index.shape[1] > 0
            }

            # -- Message passing con skip connections -----------------------
            for layer_idx, hgt in enumerate(self.hgt_layers):
                x_new = hgt(x_dict, edge_index_dict)
                for ntype in x_dict:
                    if x_new.get(ntype) is not None:
                        x_dict[ntype] = self.layer_norms[layer_idx][ntype](
                            x_dict[ntype] + self.dropout(x_new[ntype])
                        )

            # Guardar embeddings per-nodo de este timestep
            banco_embs.append(x_dict['banco'])      # (N_BANCOS, hidden_gnn)
            coop_embs.append(x_dict['cooperativa'])  # (N_COOPS, hidden_gnn)

            # -- Pooling para contexto global -------------------------------
            mask_b = snap['banco'].mask.to(device)
            mask_c = snap['cooperativa'].mask.to(device)
            pool_b = self._pool_masked(x_dict['banco'], mask_b)
            pool_c = self._pool_masked(x_dict['cooperativa'], mask_c)
            macro_emb = self.macro_mlp(snap['macro'].to(device))
            global_repr = torch.cat([pool_b, pool_c, macro_emb])
            global_repr_seq.append(global_repr)

        # -- LSTM per-nodo --------------------------------------------------
        # Apilar timesteps: (T, N_BANCOS, hidden_gnn) -> (N_BANCOS, T, hidden_gnn)
        banco_seq = torch.stack(banco_embs, dim=0).permute(1, 0, 2)   # (N_BANCOS, T, H_gnn)
        coop_seq  = torch.stack(coop_embs, dim=0).permute(1, 0, 2)    # (N_COOPS, T, H_gnn)

        # Concatenar bancos y coops como un solo batch para el LSTM compartido
        # (N_BANCOS + N_COOPS, T, hidden_gnn)
        all_nodes_seq = torch.cat([banco_seq, coop_seq], dim=0)
        node_lstm_out, _ = self.lstm_node(all_nodes_seq)    # (N_total, T, hidden_lstm_node)
        node_last = node_lstm_out[:, -1, :]                 # (N_total, hidden_lstm_node)

        h_bancos = node_last[:self.n_bancos]    # (N_BANCOS, hidden_lstm_node)
        h_coops  = node_last[self.n_bancos:]    # (N_COOPS, hidden_lstm_node)

        # -- LSTM global ----------------------------------------------------
        global_input = torch.stack(global_repr_seq, dim=0).unsqueeze(0)  # (1, T, D_global)
        global_out, _ = self.lstm_global(global_input)
        h_global = global_out[:, -1, :].squeeze(0)  # (hidden_lstm_global,)

        # -- Prediccion per-nodo --------------------------------------------
        # Concatenar hidden individual + contexto global
        h_global_b = h_global.unsqueeze(0).expand(self.n_bancos, -1)
        h_global_c = h_global.unsqueeze(0).expand(self.n_coops, -1)

        logits_banco = self.fc_banco(torch.cat([h_bancos, h_global_b], dim=1)).squeeze(-1)
        logits_coop  = self.fc_coop(torch.cat([h_coops, h_global_c], dim=1)).squeeze(-1)

        return logits_banco, logits_coop


# --- Instanciar modelo ----------------------------------------------------
model = HeteroGraphLSTM_v2(
    dim_banco          = len(FEATURES_NODO_BANCO),
    dim_coop           = len(FEATURES_NODO_COOP),
    dim_macro          = DIM_MACRO,
    hidden_gnn         = 32,
    n_gnn_layers       = 2,
    n_heads            = 4,
    hidden_lstm_node   = 64,
    hidden_lstm_global = 64,
    n_lstm_layers      = 2,
    hidden_macro       = 16,
    dropout            = 0.2,
    n_bancos           = N_BANCOS,
    n_coops            = N_COOPS,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
lstm_node_params = sum(p.numel() for p in model.lstm_node.parameters())
lstm_global_params = sum(p.numel() for p in model.lstm_global.parameters())

print(f'Modelo instanciado en {device}')
print(f'Parametros entrenables: {total_params:,}')
print(f'  LSTM per-nodo  : {lstm_node_params:,} (compartido entre todos los nodos)')
print(f'  LSTM global    : {lstm_global_params:,}')
print(f'Salida: logits -> sigmoid -> prediccion en [0,1]')
print(f'Perdida: BCE with logits')
print()
print(model)


In [ ]:
# --- Test rapido de forward pass ------------------------------------------
model.eval()
with torch.no_grad():
    seq_test = [snapshots[f] for f in periodos[:SEQ_LEN]]
    logits_b, logits_c = model(seq_test)
    pred_b = torch.sigmoid(logits_b)
    pred_c = torch.sigmoid(logits_c)
    print(f'Logits banco       : {logits_b.shape}  ->  esperado ({N_BANCOS},)')
    print(f'Logits cooperativa : {logits_c.shape}  ->  esperado ({N_COOPS},)')
    print(f'Pred banco rango   : [{pred_b.min():.4f}, {pred_b.max():.4f}]')
    print(f'Pred coop rango    : [{pred_c.min():.4f}, {pred_c.max():.4f}]')
print('Forward pass exitoso')


## Entrenamiento con Early Stopping

In [ ]:
import time

EPOCHS        = 100
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 15

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=7
)


def masked_bce(logits, target, mask):
    """
    Binary Cross-Entropy con logits, solo sobre nodos activos.
    Usa bce_with_logits que aplica sigmoid internamente
    (mas estable numericamente que sigmoid + bce por separado).
    """
    if mask.sum() == 0:
        return torch.tensor(0.0, device=device)
    return F.binary_cross_entropy_with_logits(logits[mask], target[mask])


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    n_samples  = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            seq_snapshots, target_snap = batch[0]

            logits_b, logits_c = model(seq_snapshots)

            y_b    = target_snap['banco'].y.to(device)
            y_c    = target_snap['cooperativa'].y.to(device)
            mask_b = target_snap['banco'].mask.to(device)
            mask_c = target_snap['cooperativa'].mask.to(device)

            loss_b = masked_bce(logits_b, y_b, mask_b)
            loss_c = masked_bce(logits_c, y_c, mask_c)
            loss   = loss_b + loss_c

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            n_samples  += 1

    return total_loss / max(n_samples, 1)


# --- Loop de entrenamiento ------------------------------------------------
history = {'train': [], 'val': []}
best_val_loss = float('inf')
best_epoch    = 0
patience_count = 0
CKPT_PATH = '/content/best_model_v2.pt'

print(f'{"Epoch":>6} | {"Train BCE":>10} | {"Val BCE":>10} | {"Tiempo":>8}')
print('-' * 45)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, train=True)
    val_loss   = run_epoch(val_loader,   train=False)
    elapsed    = time.time() - t0

    history['train'].append(train_loss)
    history['val'].append(val_loss)
    scheduler.step(val_loss)

    print(f'{epoch:6d} | {train_loss:10.5f} | {val_loss:10.5f} | {elapsed:7.1f}s')

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_epoch     = epoch
        patience_count = 0
        torch.save(model.state_dict(), CKPT_PATH)
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\nEarly stopping en epoca {epoch}. Mejor epoca: {best_epoch}')
            break

print(f'\nMejor val BCE: {best_val_loss:.5f} (epoca {best_epoch})')


In [ ]:
# ─── Curvas de aprendizaje ────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train'], label='Train BCE', color='#E74C3C')
ax.plot(history['val'],   label='Val BCE',   color='#3498DB')
ax.axvline(best_epoch - 1, color='gray', linestyle='--', alpha=0.7, label=f'Mejor época ({best_epoch})')
ax.set_xlabel('Época')
ax.set_ylabel('BCE Loss')
ax.set_title('Curvas de aprendizaje (Binary Cross-Entropy)')
ax.legend()
plt.tight_layout()
plt.show()


## Evaluación en test set

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, log_loss

model.load_state_dict(torch.load(CKPT_PATH))
model.eval()

all_pred_b, all_true_b, all_pred_c, all_true_c = [], [], [], []

with torch.no_grad():
    for batch in test_loader:
        seq_snapshots, target_snap = batch[0]
        logits_b, logits_c = model(seq_snapshots)

        # Aplicar sigmoid para obtener predicciones en [0,1]
        pred_b = torch.sigmoid(logits_b).cpu().numpy()
        pred_c = torch.sigmoid(logits_c).cpu().numpy()

        mask_b = target_snap['banco'].mask.numpy()
        mask_c = target_snap['cooperativa'].mask.numpy()

        # Target ya está en [0,1], no necesita inverse_transform
        true_b = target_snap['banco'].y.numpy()
        true_c = target_snap['cooperativa'].y.numpy()

        all_pred_b.append(pred_b[mask_b])
        all_true_b.append(true_b[mask_b])
        all_pred_c.append(pred_c[mask_c])
        all_true_c.append(true_c[mask_c])

pred_b_all = np.concatenate(all_pred_b)
true_b_all = np.concatenate(all_true_b)
pred_c_all = np.concatenate(all_pred_c)
true_c_all = np.concatenate(all_true_c)


def metricas(pred, true, nombre):
    # Clamp predicciones para evitar log(0)
    pred_clamped = np.clip(pred, 1e-7, 1 - 1e-7)

    mse  = np.mean((pred - true) ** 2)
    mae  = mean_absolute_error(true, pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(true, pred)

    # Log-loss (BCE): -[y*log(p) + (1-y)*log(1-p)]
    bce = -np.mean(true * np.log(pred_clamped) + (1 - true) * np.log(1 - pred_clamped))

    print(f'── {nombre} ──')
    print(f'  BCE (log-loss): {bce:.5f}')
    print(f'  MSE           : {mse:.5f}')
    print(f'  RMSE          : {rmse:.5f}')
    print(f'  MAE           : {mae:.5f}')
    print(f'  R²            : {r2:.4f}')
    print()


metricas(pred_b_all, true_b_all, 'Bancos')
metricas(pred_c_all, true_c_all, 'Cooperativas')
metricas(
    np.concatenate([pred_b_all, pred_c_all]),
    np.concatenate([true_b_all, true_c_all]),
    'Total'
)


In [ ]:
# ─── Predicho vs Real ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, true, titulo, color in [
    (axes[0], pred_b_all, true_b_all, 'Bancos',       '#E74C3C'),
    (axes[1], pred_c_all, true_c_all, 'Cooperativas', '#3498DB'),
]:
    ax.scatter(true, pred, alpha=0.4, color=color, s=15)
    lim = [min(true.min(), pred.min()), max(true.max(), pred.max())]
    ax.plot(lim, lim, 'k--', linewidth=1, label='Predicción perfecta')
    ax.set_xlabel('Morosidad real')
    ax.set_ylabel('Morosidad predicha')
    ax.set_title(f'{titulo} — Predicho vs Real')
    r2 = r2_score(true, pred)
    ax.text(0.05, 0.92, f'R² = {r2:.3f}', transform=ax.transAxes, fontsize=11)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Guardar modelo en Drive ----------------------------------------------
import shutil

SAVE_PATH = DRIVE_PATH + 'modelo_gnn_lstm_financiero_v2.pt'
shutil.copy(CKPT_PATH, SAVE_PATH)
print(f'Modelo guardado en Drive: {SAVE_PATH}')


## Guia de hiperparametros para ajustar

| Hiperparametro | Valor actual | Que controla |
|---|---|---|
| `SEQ_LEN` | 6 | Meses de historia para la LSTM. Prueba 3, 6, 12 |
| `K_INTRA` | 5 | Vecinos k-NN dentro del mismo tipo |
| `K_INTER` | 3 | Vecinos k-NN entre banco y cooperativa |
| `hidden_gnn` | 32 | Dimension del espacio latente GNN |
| `n_gnn_layers` | 2 | Profundidad del GNN |
| `n_heads` | 4 | Cabezas de atencion en HGT |
| `hidden_lstm_node` | 64 | Dimension oculta del LSTM per-nodo |
| `hidden_lstm_global` | 64 | Dimension oculta del LSTM de contexto global |
| `n_lstm_layers` | 2 | Capas LSTM (aplica a ambos LSTMs) |
| `hidden_macro` | 16 | Dimension del embedding macro |
| `dropout` | 0.2 | Regularizacion |
| `LR` | 1e-3 | Learning rate |
| `PATIENCE` | 15 | Early stopping |
